# 🏠 Nepal Real Estate — Data Cleaning Notebook

This notebook cleans **3 separate datasets** one by one:

| # | File | Type |
|---|------|------|
| 1 | `hamrobazaar_land_for_sale_kathmandu.csv` | Land listings (Hamrobazaar) |
| 2 | `nepali_land_data.csv` | Land listings (various sources) |
| 3 | `Nepali_house_dataset.csv` | House listings |

Each file is cleaned separately and saved as a new CSV.

---

## 📦 Step 0: Import Libraries

In [1]:
import pandas as pd
import re
import ast

print('Libraries loaded!')

Libraries loaded!


---
# 📁 FILE 1: hamrobazaar_land_for_sale_kathmandu.csv

**Issues to fix:**
- `Type` column is corrupted → Drop it
- `Road_Type` and `Road_Size` are messy → Extract clean road size category
- `Price` has 0 values → Remove those rows
- `location` has mixed Nepali/English → Extract district name
- `Land_Size` is in Aana unit → Keep as-is, rename clearly

In [2]:
# ── Load the file ─────────────────────────────────────────────
df1 = pd.read_csv('hamrobazaar_land_for_sale_kathmandu.csv')
print('Shape:', df1.shape)
print('Columns:', df1.columns.tolist())
df1.head()

Shape: (3869, 8)
Columns: ['ad_id', 'Type', 'Title', 'location', 'Price', 'Land_Size', 'Road_Type', 'Road_Size']


,ad_id,Type,Title,location,Price,Land_Size,Road_Type,Road_Size
0,HB-449254,Land - Commercial Use,commercial land for sale in tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",7000000.0,3.0,Pitched,Above 20 Ft.
1,HB-DC6CF2,LandIndividual,Land for sale near kmc hospital,"Kmc Hospital, Changunarayan-2, Bhaktapur",6500000.0,3.0,Others,ThirteenToTwentyFeet
2,HB-72A275,Land - Individual,10 Aana Land On Sale At Shivapuri Colony.,"बूढानीलकण्ठ, बूढानीलकण्ठ नगरपालिका, काठमाडौं ज...",5300000.0,10.0,Pitched,13 - 20 Ft.
3,HB-6CE9DF,3.5 Aana,Land on sale,"Godawari Bridge, Gwarko-Lamatar, Mahalaxmi-8, ...",3000000.0,3.5,111.29 sq. meter,LandIndividual
4,HB-5E933A,Land - Individual,15 Aana Land On Sale Near Lemon Tree Hotel.,"बूढानीलकण्ठ, बूढानीलकण्ठ नगरपालिका, काठमाडौं ज...",4500000.0,15.0,Pitched,13 - 20 Ft.


In [3]:
# ── Step 1: Drop the corrupted 'Type' column ──────────────────
df1 = df1.drop(columns=['Type'])
print('Remaining columns:', df1.columns.tolist())

Remaining columns: ['ad_id', 'Title', 'location', 'Price', 'Land_Size', 'Road_Type', 'Road_Size']


In [4]:
# ── Step 2: Remove rows where Price = 0 ───────────────────────
before = len(df1)
df1 = df1[df1['Price'] > 0]   # Keep only rows where price > 0
print(f'Removed {before - len(df1)} zero-price rows. Remaining: {len(df1)}')

Removed 1 zero-price rows. Remaining: 3868


In [ ]:
# ── Step 3: Clean Road_Type and Road_Size columns ─────────────
# Road_Size has values like 'Above 20 Ft.', '13 - 20 Ft.', '9 - 12 Ft.'
# but also garbage like location names. We extract just the clean category.

def clean_road_size(value):
    """
    Returns a clean road size category, or None if value is garbage.
    """
    if pd.isna(value): return None
    v = str(value).lower().strip()
    if 'above 20' in v or 'abovetwenty' in v: return 'Above 20 Ft'
    elif '13' in v and ('20' in v or 'twenty' in v): return '13-20 Ft'
    elif '9' in v and '12' in v: return '9-12 Ft'
    elif 'pitched' in v: return 'Pitched Road'
    elif 'gravel' in v: return 'Gravel Road'
    elif 'full' in v: return 'Full Road'
    return None   # garbage → becomes NaN

# Try Road_Size first, fill any None with Road_Type
df1['road_size_clean'] = df1['Road_Size'].apply(clean_road_size)
df1['road_size_clean'] = df1['road_size_clean'].fillna(df1['Road_Type'].apply(clean_road_size))

# Drop the original messy columns
df1 = df1.drop(columns=['Road_Type', 'Road_Size'])

print('Road size categories found:')
print(df1['road_size_clean'].value_counts(dropna=False))

In [10]:
df1.sample(4)

,ad_id,Title,location,Price,Land_Size,road_size_clean
2529,HB-2539EB,22 Aana land for sale Balkot Height,"Kathmandu City, Kathmandu",110000000.0,22.0,Pitched Road
539,HB-EA9DBE,Land for Sale at Gamcha,"Gamcha Pukhu गाम्चा पुखु, Kirtipur-9, Kathmandu",3000000.0,4.0,Pitched Road
2336,HB-3B38B2,22 aana land on sale at main track dillibazar,"डिल्लिबजार सडक, घट्टेकुलो, Kathmandu-30, काठमा...",13000000.0,22.0,Above 20 Ft
3117,HB-B88640,Land on sale,"gairedhara gharidhara, Gairidhara, Kathmandu",7500000.0,15.0,Pitched Road


In [11]:
# ── Step 4: Extract district from location ────────────────────
# Locations mix English and Nepali. We look for known district names.

def extract_district(location):
    """Returns Kathmandu, Lalitpur, Bhaktapur, or Unknown."""
    if pd.isna(location): return 'Unknown'
    location = str(location)
    for district in ['Kathmandu', 'Lalitpur', 'Bhaktapur']:
        if district.lower() in location.lower():
            return district
    return 'Unknown'

df1['district'] = df1['location'].apply(extract_district)

print('District distribution:')
print(df1['district'].value_counts())

District distribution:
district
Kathmandu    2289
Lalitpur     1035
Unknown       384
Bhaktapur     160
Name: count, dtype: int64


In [12]:
df1.sample(2)

,ad_id,Title,location,Price,Land_Size,road_size_clean,district
2050,HB-F2A6BE,5 ANA LAND FOR SALE SUNAKOTHI LALITPUR,"Kalapa Tan Marg, Sunakothi, Lalitpur-26, Satit...",3500000.0,4.499635,None,Lalitpur
1123,HB-016765,Kapan Paiyutar/Paiyatar land for sale ( 4-6 anna),"Kapan Paiyutar, Budhanilakantha-11, Kathmandu",4500000.0,4.000000,Above 20 Ft,Kathmandu


In [13]:
# ── Step 5: Rename columns to clean standard names ────────────
df1 = df1.rename(columns={
    'Title'     : 'title',
    'location'  : 'location_raw',
    'Price'     : 'price_npr',
    'Land_Size' : 'land_size_aana'
})
print('Final columns:', df1.columns.tolist())

Final columns: ['ad_id', 'title', 'location_raw', 'price_npr', 'land_size_aana', 'road_size_clean', 'district']


In [14]:
# ── Step 6: Final check ───────────────────────────────────────
print('=== FILE 1 FINAL SUMMARY ===')
print(f'Total rows: {len(df1)}')
print(f'Total columns: {len(df1.columns)}')
print('\nNull values per column:')
print(df1.isnull().sum())
df1.head()

=== FILE 1 FINAL SUMMARY ===
Total rows: 3868
Total columns: 7

Null values per column:
ad_id                0
title                0
location_raw         0
price_npr            0
land_size_aana       0
road_size_clean    807
district             0
dtype: int64


,ad_id,title,location_raw,price_npr,land_size_aana,road_size_clean,district
0,HB-449254,commercial land for sale in tokha,"Tokha-3, Kathmandu, Bagmati Pradesh",7000000.0,3.0,Above 20 Ft,Kathmandu
1,HB-DC6CF2,Land for sale near kmc hospital,"Kmc Hospital, Changunarayan-2, Bhaktapur",6500000.0,3.0,None,Bhaktapur
2,HB-72A275,10 Aana Land On Sale At Shivapuri Colony.,"बूढानीलकण्ठ, बूढानीलकण्ठ नगरपालिका, काठमाडौं ज...",5300000.0,10.0,13-20 Ft,Unknown
3,HB-6CE9DF,Land on sale,"Godawari Bridge, Gwarko-Lamatar, Mahalaxmi-8, ...",3000000.0,3.5,None,Lalitpur
4,HB-5E933A,15 Aana Land On Sale Near Lemon Tree Hotel.,"बूढानीलकण्ठ, बूढानीलकण्ठ नगरपालिका, काठमाडौं ज...",4500000.0,15.0,13-20 Ft,Unknown


In [17]:
# ── Step 7: Save cleaned file ─────────────────────────────────
df1.to_csv('cleaned_hamrobazaar_land.csv', index=False)
print('Saved: cleaned_hamrobazaar_land.csv ✅')

Saved: cleaned_hamrobazaar_land.csv ✅


---
# 📁 FILE 2: nepali_land_data.csv

**Issues to fix:**
- `Price` has many formats: 'Rs. 22 Lac/aana', 'Rs. 1.45 Cr', 'Price on call', etc.
  → Extract numeric NPR value + store the price unit separately
- `Road_Access` like '20 Feet /Soil Stabilized' → Split into width (feet) + surface type
- `Land_AREA` like '3.1 aana' → Extract numeric value
- Lots of missing Land_AREA (63%) → Keep as NaN
- `Facing_Direction` → Rename and keep

In [18]:
# ── Load the file ─────────────────────────────────────────────
df2 = pd.read_csv('nepali_land_data.csv')
print('Shape:', df2.shape)
print('Columns:', df2.columns.tolist())
df2.head()

Shape: (2043, 5)
Columns: ['Price', 'Location', 'Road_Access', 'Facing_Direction', 'Land_AREA']


,Price,Location,Road_Access,Facing_Direction,Land_AREA
0,Rs. 22 Lac/aana,"Duwakot, Bhaktapur",20 Feet /Soil Stabilized,West,3.1 aana
1,Rs. 38 Lac/aana,"Sitapakha, Lalitpur",13 Feet /Dhalan,West,NaN
2,Rs. 40 Lac/aana,"Changathali, Lalitpur",20 Feet /Dhalan,South,NaN
3,Rs. 35 Lac/aana,"Gothatar, Kathmandu",13 Feet /Dhalan,North,3.3 aana
4,Rs. 45 Lac/aana,"Shital Height, Lalitpur",20 Feet /Dhalan,North-East,NaN


In [23]:
# ── Step 1: Clean Price column ───────────────────────────────
# Price formats found in this file:
#   'Rs. 22 Lac/aana'   → 2,200,000 NPR per aana
#   'Rs. 1.45 Cr'       → 14,500,000 NPR total
#   'Rs. 43 Lac'        → 4,300,000 NPR total
#   'Rs. 60,000 /m'     → 60,000 NPR per sq meter
#   'Price on call'     → unknown → NaN
#
# We store TWO columns:
#   price_amount_npr  : the numeric value in NPR
#   price_unit        : what it is priced per (per_aana, total, per_dhur, etc.)

def extract_price_info(price_str):
    """Extracts numeric NPR value and unit from messy price string."""
    if pd.isna(price_str) or 'call' in str(price_str).lower():
        return {'price_amount_npr': None, 'price_unit': None}
    
    price_str = str(price_str).strip()
    
    # Remove commas (e.g. '60,000' → '60000')
    price_clean = price_str.replace(',', '')
    
    # \d+\.?\d* matches numbers like 22, 22.5, 1.45
    # This avoids accidentally matching lone '.' from 'Rs.'
    match = re.search(r'\d+\.?\d*', price_clean)
    if not match:
        return {'price_amount_npr': None, 'price_unit': None}
    
    number = float(match.group())
    
    # Determine the multiplier (Crore = 10M, Lac = 100K)
    if 'Cr' in price_str:    multiplier = 10_000_000
    elif 'Lac' in price_str: multiplier = 100_000
    else:                    multiplier = 1
    
    amount_npr = number * multiplier
    
    # Determine the unit
    if '/aana' in price_str:     unit = 'per_aana'
    elif '/dhur' in price_str:   unit = 'per_dhur'
    elif '/ropani' in price_str: unit = 'per_ropani'
    elif '/kattha' in price_str: unit = 'per_kattha'
    elif '/m' in price_str:      unit = 'per_sqm'
    elif '/y' in price_str:      unit = 'per_year'
    else:                         unit = 'total'
    
    return {'price_amount_npr': amount_npr, 'price_unit': unit}

# Apply and expand to two columns
price_info = df2['Price'].apply(extract_price_info)
df2['price_amount_npr'] = price_info.apply(lambda x: x['price_amount_npr'])
df2['price_unit']       = price_info.apply(lambda x: x['price_unit'])

print('Price unit distribution:')
print(df2['price_unit'].value_counts(dropna=False))
print('\nSample:')
print(df2[['Price', 'price_amount_npr', 'price_unit']].sample(8))

Price unit distribution:
price_unit
per_aana      1797
None            99
total           95
per_dhur        23
per_sqm         16
per_ropani       6
per_kattha       6
per_year         1
Name: count, dtype: int64

Sample:
                Price  price_amount_npr price_unit
1858  Rs. 45 Lac/aana         4500000.0   per_aana
1113  Rs. 36 Lac/aana         3600000.0   per_aana
183   Rs. 58 Lac/aana         5800000.0   per_aana
748   Rs. 55 Lac/aana         5500000.0   per_aana
1613  Rs. 23 Lac/aana         2300000.0   per_aana
2001       Rs. 80 Lac         8000000.0      total
385   Rs. 45 Lac/aana         4500000.0   per_aana
1986  Rs. 38 Lac/aana         3800000.0   per_aana


In [30]:
# ── Step 2: Clean Road_Access column ─────────────────────────
# '20 Feet /Soil Stabilized' → road_width_feet=20, road_surface='Soil Stabilized'

def extract_road_width(road_str):
    """Extracts road width in feet."""
    if pd.isna(road_str): return None
    m = re.search(r'(\d+)\s*Feet', str(road_str), re.IGNORECASE)
    return int(m.group(1)) if m else None

def extract_road_surface(road_str):
    """Extracts road surface type (part after '/')."""
    if pd.isna(road_str): return None
    parts = str(road_str).split('/')
    return parts[1].strip() if len(parts) > 1 else None

df2['road_width_feet'] = df2['Road_Access'].apply(extract_road_width)
df2['road_surface']    = df2['Road_Access'].apply(extract_road_surface)

print('Sample road access cleaning:')
print(df2[['Road_Access', 'road_width_feet', 'road_surface']].sample(8))

Sample road access cleaning:
                Road_Access  road_width_feet  road_surface
1832        25 Feet /Dhalan             25.0        Dhalan
1034         22 Feet /Paved             22.0         Paved
1157     13 Feet /Gravelled             13.0     Gravelled
854          13 Feet /Paved             13.0         Paved
1203         13 Feet /Paved             13.0         Paved
1433  13 Feet /Black Topped             13.0  Black Topped
2002     12 Feet /Gravelled             12.0     Gravelled
923         13 Feet /Dhalan             13.0        Dhalan


In [31]:
df2['Land_AREA'].isnull().sum()

np.int64(1282)

In [29]:
# ── Step 3: Clean Land_AREA column ───────────────────────────
# '3.1 aana' → 3.1

def extract_land_area(area_str):
    """Extracts numeric value from land area string."""
    if pd.isna(area_str): return None
    m = re.search(r'\d+\.?\d*', str(area_str))
    return float(m.group()) if m else None

df2['land_size_aana'] = df2['Land_AREA'].apply(extract_land_area)

print('Sample:')
print(df2[['Land_AREA', 'land_size_aana']].dropna().head(8))
print(f'\nMissing land area: {df2["land_size_aana"].isna().sum()} rows (kept as NaN)')

Sample:
   Land_AREA  land_size_aana
0   3.1 aana             3.1
3   3.3 aana             3.3
6   8.2 aana             8.2
9   3.2 aana             3.2
13  5.1 aana             5.1
17  3.1 aana             3.1
18  3.1 aana             3.1
20  3.1 aana             3.1

Missing land area: 1282 rows (kept as NaN)


In [32]:
# ── Step 4: Rename, clean up, extract district ────────────────
df2 = df2.rename(columns={
    'Location'         : 'location_raw',
    'Facing_Direction' : 'facing'
})

# Drop original raw columns now that we have clean versions
df2 = df2.drop(columns=['Price', 'Road_Access', 'Land_AREA'])

# Extract district (same function as File 1)
def extract_district(location):
    if pd.isna(location): return 'Unknown'
    location = str(location)
    for d in ['Kathmandu', 'Lalitpur', 'Bhaktapur']:
        if d.lower() in location.lower(): return d
    return 'Unknown'

df2['district'] = df2['location_raw'].apply(extract_district)
print('Final columns:', df2.columns.tolist())

Final columns: ['location_raw', 'facing', 'price_amount_npr', 'price_unit', 'road_width_feet', 'road_surface', 'land_size_aana', 'district']


In [33]:
# ── Step 5: Final check ───────────────────────────────────────
print('=== FILE 2 FINAL SUMMARY ===')
print(f'Total rows: {len(df2)}')
print(f'Total columns: {len(df2.columns)}')
print('\nNull values per column:')
print(df2.isnull().sum())
df2.head()

=== FILE 2 FINAL SUMMARY ===
Total rows: 2043
Total columns: 8

Null values per column:
location_raw           0
facing                42
price_amount_npr      99
price_unit            99
road_width_feet       96
road_surface          75
land_size_aana      1282
district               0
dtype: int64


,location_raw,facing,price_amount_npr,price_unit,road_width_feet,road_surface,land_size_aana,district
0,"Duwakot, Bhaktapur",West,2200000.0,per_aana,20.0,Soil Stabilized,3.1,Bhaktapur
1,"Sitapakha, Lalitpur",West,3800000.0,per_aana,13.0,Dhalan,NaN,Lalitpur
2,"Changathali, Lalitpur",South,4000000.0,per_aana,20.0,Dhalan,NaN,Lalitpur
3,"Gothatar, Kathmandu",North,3500000.0,per_aana,13.0,Dhalan,3.3,Kathmandu
4,"Shital Height, Lalitpur",North-East,4500000.0,per_aana,20.0,Dhalan,NaN,Lalitpur


In [34]:
# ── Step 6: Save cleaned file ─────────────────────────────────
df2.to_csv('cleaned_nepali_land.csv', index=False)
print('Saved: cleaned_nepali_land.csv ✅')

Saved: cleaned_nepali_land.csv ✅


---
# 📁 FILE 3: Nepali_house_dataset.csv

**Issues to fix:**
- `PRICE` has mixed formats: 'Rs. 2.9 Cr', 'Rs. 12000000' → Normalize to NPR
- `LAND AREA` like '4.0 aana' → Extract numeric
- `BUILDUP AREA` like '3600 Sq. Ft.' but '-1' and '0' exist → Clean and remove invalid
- `ROAD ACCESS` like '12 Feet' → Extract number
- `FACING` has 36 messy variations of 8 directions → Standardize
- `BUILT YEAR` is Bikram Sambat (e.g. '2076 B.S') → Keep + add AD equivalent
- `PARKING` like '1 CaRs.  & 2 Bikes' → Extract car count and bike count
- `BEDROOM`/`BATHROOM` values > 20 are data errors → Replace with NaN
- `AMENITIES` is string-formatted list → Convert to clean comma-separated string

In [35]:
# ── Load the file ─────────────────────────────────────────────
df3 = pd.read_csv('Nepali_house_dataset.csv')
print('Shape:', df3.shape)
print('Columns:', df3.columns.tolist())
df3.head()

Shape: (3418, 13)
Columns: ['TITLE', 'LOCATION', 'PRICE', 'LAND AREA', 'BUILDUP AREA', 'ROAD ACCESS', 'FACING', 'FLOOR', 'BEDROOM', 'BATHROOM', 'BUILT YEAR', 'PARKING', 'AMENITIES']


,TITLE,LOCATION,PRICE,LAND AREA,BUILDUP AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
0,House for Sale,"Imadol, Lalitpur",Rs. 2.9 Cr,4.0 aana,NaN,12 Feet,West,3.0,5.0,4.0,2076 B.S,1 CaRs. & 2 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
1,House for Sale,"Satdobato, Lalitpur",Rs. 4.75 Cr,3.0 aana,NaN,10 Feet,West,4.5,5.0,6.0,2076 B.S,2 CaRs. & 2 Bikes,"['Earthquake Resistant', 'Parquet', 'Drinking ..."
2,4 BHK House for Sale,"Imadol, Lalitpur",Rs. 1.99 Cr,2.3 aana,NaN,10 Feet,West,2.5,4.0,4.0,2060 B.S,1 CaRs. & 3 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
3,Bungalow House for Sale,"Bhaisepati, Lalitpur",Rs. 4 Cr,7.0 aana,NaN,12 Feet,North-West,2.5,4.0,3.0,2059 B.S,4 CaRs. & 4 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
4,House for Rent,"Maharajgunj, Kathmandu",Rs. 12000000,6.0 aana,NaN,20 Feet,South,2.0,4.0,4.0,2071 B.S,4 CaRs. & 5 Bikes,"['Earthquake Resistant', 'Parquet', 'Parking',..."


In [37]:
# ── Step 1: Clean PRICE column ───────────────────────────────
# 'Rs. 2.9 Cr '   → 29,000,000 NPR
# 'Rs. 4.75 Cr '  → 47,500,000 NPR
# 'Rs. 12000000'  → 12,000,000 NPR (already in NPR)

def clean_house_price(p):
    """Converts all price formats to a single NPR number."""
    if pd.isna(p): return None
    p = str(p).strip()
    # \d+\.?\d* avoids matching lone '.' in 'Rs.'
    match = re.search(r'\d+\.?\d*', p.replace(',', ''))
    if not match: return None
    number = float(match.group())
    if 'Cr' in p:                    return number * 10_000_000
    elif 'Lac' in p or 'Lakh' in p: return number * 100_000
    else:                             return number

df3['price_npr'] = df3['PRICE'].apply(clean_house_price)

# Remove rows where price is 0 or could not be determined
before = len(df3)
df3 = df3[df3['price_npr'] > 0]
print(f'Removed {before - len(df3)} rows with invalid price.')
print('\nSample price cleaning:')
print(df3[['PRICE', 'price_npr']].sample(8))

Removed 0 rows with invalid price.

Sample price cleaning:
              PRICE   price_npr
460     Rs. 2.95 Cr  29500000.0
2550  Rs. 75,000 /m     75000.0
2118    Rs. 1.5 Cr   15000000.0
2174      Rs. 7 Cr   70000000.0
2753    Rs. 2.1 Cr   21000000.0
2941    Rs. 1.2 Cr   12000000.0
2205    Rs. 22 Lac    2200000.0
810    Rs. 3.35 Cr   33500000.0


In [48]:
pd.set_option('display.max_rows', None)
df3['LAND AREA'].value_counts().nunique

<bound method IndexOpsMixin.nunique of LAND AREA
4.0 aana            396
3.0 aana            303
3.2 aana            232
5.0 aana            217
4.2 aana            144
6.0 aana            129
3.1 aana            127
3.3 aana             87
8.0 aana             76
4.1 aana             74
7.0 aana             70
5.2 aana             67
2.2 aana             64
16.0 aana            58
9.0 aana             58
10.0 aana            56
2.3 aana             53
3 aana               46
4.3 aana             45
12.0 aana            44
3.5 aana             40
6.2 aana             36
5.1 aana             36
4 aana               33
4.5 aana             27
11.0 aana            26
5.3 aana             23
14.0 aana            23
7.2 aana             23
32.0 aana            23
6.1 aana             22
8.2 aana             20
24.0 aana            20
17.0 aana            15
5 aana               14
15.0 aana            14
2.5 aana             14
18.0 aana            13
13.0 aana            13
9.2 aana       

In [62]:
import re
import pandas as pd

def extract_land_area_to_aana(area_str):
    if pd.isna(area_str): return None
    area_str = str(area_str).lower().strip()
    
    # 1. Extract the first number found
    m = re.search(r'(\d+\.?\d*)', area_str)
    if not m: return None
    val = float(m.group(1))
    
    # 2. Conversion Logic (Standardizing to Aana)
    # 1 Kattha ≈ 10.37 Aana
    # 1 Aana = 342.25 Sq. Ft
    if 'kattha' in area_str:
        return round(val * 10.37, 2)
    elif 'sq. ft' in area_str or 'sq ft' in area_str:
        return round(val / 342.25, 2)
    elif 'sq. mtr' in area_str:
        return round(val / 31.80, 2) # 1 Aana ≈ 31.8 Sq Meters
    else:
        # Defaults to Aana if no unit is specified or if 'aana' is present
        return val

# Apply the new function
df3['land_size_aana'] = df3['LAND AREA'].apply(extract_land_area_to_aana)

# Check the results for different units
print(df3[['LAND AREA', 'land_size_aana']].head(20))

   LAND AREA  land_size_aana
0   4.0 aana             4.0
1   3.0 aana             3.0
2   2.3 aana             2.3
3   7.0 aana             7.0
4   6.0 aana             6.0
5   6.0 aana             6.0
6   3.2 aana             3.2
7   4.3 aana             4.3
8   2.2 aana             2.2
9   5.0 aana             5.0
10  5.2 aana             5.2
11  3.0 aana             3.0
12  4.0 aana             4.0
13  4.0 aana             4.0
14  6.2 aana             6.2
15  4.2 aana             4.2
16  5.0 aana             5.0
17  3.2 aana             3.2
18  5.2 aana             5.2
19  4.1 aana             4.1


In [65]:
df3.sample(20)

,TITLE,LOCATION,PRICE,LAND AREA,BUILDUP AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES,price_npr,land_size_aana
2768,House for sale or rent in Tokha,"Tokha, Kathmandu",Rs. 3.8 Cr,7.0 aana,NaN,20 Feet,West,2.5,4.0,4.0,2075 B.S,NaN,"['Garden', 'Drainage', 'Bathroom', 'Drinking W...",38000000.0,7.0
3288,New Adheshwor House On Sale,"Sitapaila, Kathmandu",Rs. 28500000,3.7 aana,2667 Sq.ft,13 Feet,NORTH-WEST,3.5,6.0,3.0,2079 B.S,1 Car & 3 Bikes,[],28500000.0,3.7
101,House for Sale,"Harisiddhi, Lalitpur",Rs. 2.5 Cr,3.1 aana,NaN,13 Feet,West,2.5,4.0,4.0,2068 B.S,1 CaRs. & 1 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ...",25000000.0,3.1
685,"House on sale at Hattigauda, Budhanilkantha","Hattigaunda, Kathmandu",Rs. 3.45 Cr,4.2 aana,NaN,13 Feet,West,2.0,5.0,5.0,2079 B.S,NaN,"['Bathroom', 'Parking', 'Drainage', 'Drinking ...",34500000.0,4.2
3036,House for rent at Dhobighat,"Dhobighat, Lalitpur",Rs. 1.75 Lac/m,NaN,NaN,20 Feet,East,NaN,5.0,NaN,2077 B.S,NaN,"['Water Supply', 'Drainage', 'Power Backup Wat...",175000.0,NaN
741,"Beautiful house for sale at Kapan, Budhanilkantha","Kapan, Kathmandu",Rs. 3.5 Cr,6.0 aana,NaN,10-12 Feet,North-West,2.5,12.0,4.0,2069 B.S,NaN,"['Drinking Water', 'Bathroom', 'Parking', 'Dra...",35000000.0,6.0
962,Residential Bungalow for rent in Golfutar,"Golfutar, Kathmandu",Rs. 1 Lac/m,8.0 aana,NaN,12-18 Feet,West,2.5,6.0,4.0,2076 B.S,NaN,"['Marbel', 'Parquet', 'Drainage', 'Drinking Wa...",100000.0,8.0
3307,Imadol New House on Sale,"Imadol, Lalitpur",Rs. 28500000,3.1 aana,2427 Sq.ft,13 Feet,EAST-SOUTH,2.0,5.0,3.0,2079 B.S,1 Car & 3 Bikes,"['Water Filter', 'Modular Kitchen', 'Water Well']",28500000.0,3.1
2772,Beautiful house on sale at Kalanki,"Kalanki, Kathmandu",Rs. 2.9 Cr,4.1 aana,2100 Sq. Feet,10 Feet,South,2.5,5.0,4.0,2075 B.S,NaN,"['Earthquake Resistant', 'Marbel', 'Power Back...",29000000.0,4.1
3314,Bhimdhunga House,"Nagarjung, Kathmndu",Rs. 20500000,3 aana,2752 sq.ft,13 Feet,NORTH,2.5,6.0,3.0,2078 B.S,1 Car & 2 Bikes,"['Power Backup', 'Modular Kitchen', 'Water Well']",20500000.0,3.0


In [67]:
df3['BUILDUP AREA'].value_counts()

BUILDUP AREA
2500 Sq. Feet                     18
0 Sq. Ft.                         15
3000 Sq. Feet                     15
2700 Sq. Feet                     12
2400 Sq. Feet                     11
5000 Sq. Feet                     11
2800 Sq. Feet                     11
2600 Sq. Feet                     10
2267 Sq.ft                        10
2320 sq.ft.                        9
3500 Sq. Feet                      8
2000 Sq. Feet                      7
2750 Sq.ft                         7
4000 Sq. Feet                      7
2200 Sq. Feet                      6
2300 Sq. Feet                      6
3158Sq.ft                          5
8000 Sq. Feet                      5
2100 Sq. Feet                      5
1500 Sq. Feet                      5
6500 Sq. Feet                      4
1900 Sq. Feet                      4
3200 Sq. Feet                      4
2500 Sq. Ft.                       4
3500 Sq. Ft.                       4
3100 sq.ft                         4
2667 Sq.ft               

In [68]:
def extract_buildup_to_sqft(area_str):
    if pd.isna(area_str): return None
    area_str = str(area_str).lower().strip()
    
    # 1. Handle RAPD Format (e.g., "0-4-0-0 Ropani-aana-Paisa-Daam")
    if 'ropani' in area_str:
        # Extract all numbers in the string
        parts = re.findall(r'\d+\.?\d*', area_str)
        if len(parts) >= 2: # We need at least Ropani and Aana
            r = float(parts[0])
            a = float(parts[1])
            p = float(parts[2]) if len(parts) > 2 else 0
            d = float(parts[3]) if len(parts) > 3 else 0
            # Conversion: 1 Ropani = 5476 sqft, 1 Aana = 342.25 sqft, 
            # 1 Paisa = 85.56 sqft, 1 Daam = 21.39 sqft
            total_sqft = (r * 5476) + (a * 342.25) + (p * 85.56) + (d * 21.39)
            return round(total_sqft, 2)

    # 2. Extract the first number for standard units
    m = re.search(r'(\d+\.?\d*)', area_str)
    if not m: return None
    val = float(m.group(1))
    if val <= 0: return None # Handle 0 or negative
    
    # 3. Handle Square Meters
    if 'sq. meter' in area_str or 'sq. mtr' in area_str or 'sq meter' in area_str:
        return round(val * 10.7639, 2) # 1 Sq. Mtr = 10.7639 Sq. Ft
    
    # 4. Handle Aana (some houses listed buildup in Aana)
    if 'aana' in area_str:
        return round(val * 342.25, 2)
    
    # 5. Handle Acres
    if 'acres' in area_str or 'acre' in area_str:
        return round(val * 43560, 2)

    # Default: Assume Sq. Ft
    return val

df3['buildup_area_sqft'] = df3['BUILDUP AREA'].apply(extract_buildup_to_sqft)

In [82]:
print(df3[['BUILDUP AREA', 'buildup_area_sqft']].sample(20))

           BUILDUP AREA  buildup_area_sqft
466                 NaN                NaN
2695                NaN                NaN
3097                NaN                NaN
1525  2574.15 Sq. Meter           27707.89
1671                NaN                NaN
2495   1474.69 Sq. Feet            1474.69
3291         1810 sq.ft            1810.00
208                 NaN                NaN
3270         3587 sq.ft            3587.00
2687      5500 Sq. Feet            5500.00
845                 NaN                NaN
2561                NaN                NaN
2454                NaN                NaN
1103                NaN                NaN
2110                NaN                NaN
552                 NaN                NaN
1055                NaN                NaN
0                   NaN                NaN
2550      2085 Sq. Feet            2085.00
1630                NaN                NaN


In [83]:
df3['ROAD ACCESS'].value_counts()

ROAD ACCESS
13 Feet         979
20 Feet         484
12 Feet         265
16 Feet         210
12-18 Feet      185
14 Feet         185
10 Feet         165
15 Feet         133
18 Feet          66
26 Feet          42
0 Feet           36
14-20 Feet       28
22 Feet          28
10-12 Feet       26
11 Feet          21
13-20 Feet       20
12-16 Feet       20
30 Feet          18
12-15 Feet       15
8 Feet           15
32 Feet          13
24 Feet          11
12-14 Feet       11
25 Feet          10
4 Feet            9
8-12 Feet         9
10-15 Feet        8
16-20 Feet        8
36 Feet           8
27 Feet           8
40 Feet           7
10-16 Feet        7
18-20 Feet        6
8-10 Feet         6
9 Feet            6
12                6
15-20 Feet        5
12-20 Feet        5
23 Feet           4
13/20 Feet        4
6 Feet            4
13-18 Feet        4
13-15 Feet        4
21 Feet           3
5 Feet            3
20 Meter          3
28 Feet           3
20-26 Feet        3
17 Feet           3
12-13 Fe

In [84]:
def extract_road_width_cleaned(road_str):
    if pd.isna(road_str): return None
    road_str = str(road_str).lower().strip()
    
    # 1. Handle Ranges (e.g., "12-18 Feet", "13/20 Feet", "12 & 22 Feet")
    # This finds all numbers in the string
    numbers = re.findall(r'(\d+\.?\d*)', road_str)
    if not numbers: return None
    
    # Convert extracted strings to floats
    nums = [float(n) for n in numbers]
    
    # Logic: If it's a range or dual access, take the average
    # (Example: 12-18 becomes 15.0)
    val = sum(nums) / len(nums)
    
    # 2. Handle Units
    # If "meter" is mentioned, convert to feet (1m ≈ 3.28ft)
    if 'meter' in road_str or 'metre' in road_str:
        val = val * 3.28
        
    # 3. Filter Outliers/Invalid
    if val <= 0: return None
    if val > 100: return None # A 200ft road is likely an error or a major highway
    
    return round(val, 1)

df3['road_width_feet'] = df3['ROAD ACCESS'].apply(extract_road_width_cleaned)

In [89]:
print(df3[['ROAD ACCESS', 'road_width_feet']].sample(20))

     ROAD ACCESS  road_width_feet
3110     12 Feet             12.0
495      20 Feet             20.0
2456     20 Feet             20.0
1627     13 Feet             13.0
787      10 Feet             10.0
621      15 Feet             15.0
2592     13 Feet             13.0
361     10 Meter             32.8
565      16 Feet             16.0
1336     20 Feet             20.0
942      18 Feet             18.0
1212  10-12 Feet             11.0
2993     12 Feet             12.0
1026     16 Feet             16.0
3367     16 Feet             16.0
2239     13 Feet             13.0
505      28 Feet             28.0
1152  10-15 Feet             12.5
1575     20 Feet             20.0
1705     20 Feet             20.0


In [91]:
def standardize_facing_refined(f):
    if pd.isna(f): return None
    
    # Lowercase and clean up for keyword searching
    f = str(f).lower().strip()
    
    # 1. Handle dual primary directions (e.g., EAST/WEST) 
    # Usually, the first one mentioned is the primary face.
    if '/' in f or 'and' in f:
        f = f.split('/')[0].split('and')[0].strip()

    # 2. Check for 8-point compass directions
    # Specific (Ordinal) directions first
    if 'north' in f and 'east' in f: return 'North-East'
    if 'north' in f and 'west' in f: return 'North-West'
    if 'south' in f and 'east' in f: return 'South-East'
    if 'south' in f and 'west' in f: return 'South-West'
    
    # 3. Simple Cardinal directions
    if 'north' in f: return 'North'
    if 'south' in f: return 'South'
    if 'east' in f:  return 'East'
    if 'west' in f:  return 'West'
    
    return None

df3['facing'] = df3['FACING'].apply(standardize_facing_refined)

In [94]:
print(df3[['FACING', 'facing']].sample(20))

          FACING      facing
1420       North       North
2123        East        East
930        South       South
50          West        West
2183       South       South
497   North-West  North-West
20    North-East  North-East
3200        EAST        East
1931        East        East
2678       North       North
95         North       North
905         East        East
116        South       South
1118         NaN        None
583        South       South
3409       North       North
2217       south       South
3134       North       North
1253       South       South
2352  North-West  North-West


In [98]:
import re
import pandas as pd
from datetime import datetime

def extract_built_year_dynamic(y):
    if pd.isna(y): return None
    
    # 1. Extract exactly 4 digits
    m = re.search(r'(\d{4})', str(y))
    if not m: return None
    
    year_bs = int(m.group(1))
    
    # 2. Validation: Ensure it falls within a realistic range for BS
    if 2000 <= year_bs <= 2100:
        return year_bs
    return None

# Calculate current years
current_year_ad = datetime.now().year
current_year_bs = current_year_ad + 56 # Standard conversion factor

# Apply cleaning
df3['built_year_bs'] = df3['BUILT YEAR'].apply(extract_built_year_dynamic)

# Standardize to AD (Approximate)
df3['built_year_ad'] = df3['built_year_bs'] - 57

# Calculate House Age based on current BS year
df3['house_age'] = current_year_bs - df3['built_year_bs']

# Quick look at the age distribution
print(f"Calculated based on current year: {current_year_ad} AD / {current_year_bs} BS")


Calculated based on current year: 2026 AD / 2082 BS
  BUILT YEAR  built_year_bs  house_age
0   2076 B.S         2076.0        6.0
1   2076 B.S         2076.0        6.0
2   2060 B.S         2060.0       22.0
3   2059 B.S         2059.0       23.0
4   2071 B.S         2071.0       11.0


In [99]:
print(df3[['BUILT YEAR', 'built_year_ad','house_age']].sample(20))

     BUILT YEAR  built_year_ad  house_age
205    2059 B.S         2002.0       23.0
999    2067 B.S         2010.0       15.0
754    2074 B.S         2017.0        8.0
965    2076 B.S         2019.0        6.0
563    2077 B.S         2020.0        5.0
2615   2068 B.S         2011.0       14.0
2834   2065 B.S         2008.0       17.0
159    2073 B.S         2016.0        9.0
1786   2070 B.S         2013.0       12.0
576    2070 B.S         2013.0       12.0
301    2058 B.S         2001.0       24.0
1878   2072 B.S         2015.0       10.0
379    2076 B.S         2019.0        6.0
2958   2070 B.S         2013.0       12.0
1820   2065 B.S         2008.0       17.0
952    2060 B.S         2003.0       22.0
2225   2058 B.S         2001.0       24.0
244    2061 B.S         2004.0       21.0
1070   2074 B.S         2017.0        8.0
2067   2070 B.S         2013.0       12.0


In [100]:
df3['PARKING'].value_counts()

PARKING
1 CaRs.  & 2 Bikes        84
1 Car & 3 Bikes           80
1 CaRs.  & 3 Bikes        32
2 Bikes                   31
1 CaRs.  & 1 Bikes        29
2 CaRs.  & 3 Bikes        29
1 Car & 2 Bikes           28
2 CaRs.  & 4 Bikes        25
3 CaRs.  & 5 Bikes        19
2 CaRs.  & 2 Bikes        19
2 CaRs. & 3 Bikes         14
2 CaRs. & 5 Bikes         12
1 Car & 4 Bikes           12
2 Car & 5 Bikes           12
4 CaRs.  & 5 Bikes        11
1 CaRs.                   10
1 Car & 5 Bikes           10
3 CaRs.  & 4 Bikes         9
1 CaRs.  & 4 Bikes         8
2 CaRs.  & 5 Bikes         8
2 CaRs.                    8
3 Bikes                    7
4 CaRs.  & 4 Bikes         6
1 Bikes                    6
3 CaRs.  & 3 Bikes         6
1 CaRs.  & 5 Bikes         5
3 CaRs.                    5
3 CaRs. & 4 Bikes          5
5 CaRs.  & 5 Bikes         4
9 CaRs.  & 9 Bikes         4
2 CaRs. & 8 Bikes          4
2 Car & 3 Bikes            4
2 CaRs.  & 1 Bikes         4
2 CaRs.  & 8 Bikes         3
4 Bike

In [101]:
def extract_parking_details(p):
    if pd.isna(p): return pd.Series([None, None])
    p = str(p).lower().strip()
    
    # 1. Handle Excel Date Errors (e.g., "13-Nov" often means 1-3 cars/bikes)
    # We'll treat these as None because they are corrupted data points.
    if '-nov' in p or '-dec' in p:
        return pd.Series([None, None])

    # 2. Extract Cars
    # Look for a number followed by 'ca' (cars) or just the start of the string if it's just a number
    cars_match = re.search(r'(\d+)\s*(?:car|ca)', p)
    cars = int(cars_match.group(1)) if cars_match else None
    
    # 3. Extract Bikes
    # Look for a number followed by 'bike' or 'motercycle'
    bikes_match = re.search(r'(\d+)\s*(?:bike|motercycle)', p)
    bikes = int(bikes_match.group(1)) if bikes_match else None
    
    # 4. Handle Case where it's just a single number (e.g., "8" or "1")
    # If both are None and the whole string is just a number, assume it's cars (common in real estate)
    if cars is None and bikes is None:
        if p.isdigit():
            cars = int(p)

    return pd.Series([cars, bikes])

# Apply to create two columns at once
df3[['parking_cars', 'parking_bikes']] = df3['PARKING'].apply(extract_parking_details)

In [105]:
print(df3[['PARKING','parking_cars', 'parking_bikes']])

                     PARKING  parking_cars  parking_bikes
0         1 CaRs.  & 2 Bikes           1.0            2.0
1         2 CaRs.  & 2 Bikes           2.0            2.0
2         1 CaRs.  & 3 Bikes           1.0            3.0
3         4 CaRs.  & 4 Bikes           4.0            4.0
4         4 CaRs.  & 5 Bikes           4.0            5.0
5         4 CaRs.  & 5 Bikes           4.0            5.0
6         4 CaRs.  & 5 Bikes           4.0            5.0
7         1 CaRs.  & 2 Bikes           1.0            2.0
8         1 CaRs.  & 3 Bikes           1.0            3.0
9         2 CaRs.  & 2 Bikes           2.0            2.0
10                       NaN           NaN            NaN
11                   2 Bikes           NaN            2.0
12                   3 Bikes           NaN            3.0
13                       NaN           NaN            NaN
14                   3 CaRs.           3.0            NaN
15        1 CaRs.  & 2 Bikes           1.0            2.0
16        1 Ca

In [106]:
# 1. Convert to numeric and force errors to NaN
df3['bedrooms'] = pd.to_numeric(df3['BEDROOM'], errors='coerce')
df3['bathrooms'] = pd.to_numeric(df3['BATHROOM'], errors='coerce')

# 2. Define your "Residential Bounds"
# Max 10 for a house is safe. Max 15 for a "Big House/Hostel".
# Anything over 15 is almost certainly a data entry error (e.g., someone typing 40 instead of 4).
bedroom_limit = 15
bathroom_limit = 12

# 3. Clean them!
# Instead of deleting the row, we nullify the specific impossible values 
# so we can still use the 'Price' and 'Location' data for other analysis.
df3.loc[df3['bedrooms'] > bedroom_limit, 'bedrooms'] = None
df3.loc[df3['bathrooms'] > bathroom_limit, 'bathrooms'] = None

# 4. Handle the "0" values (A house cannot have 0 bedrooms)
df3.loc[df3['bedrooms'] <= 0, 'bedrooms'] = None

In [113]:
# ── Step 9: Clean AMENITIES column ───────────────────────────
# AMENITIES is stored as a string that looks like a Python list:
# "['Earthquake Resistant', 'Marble', 'Parking', ...]"
# We use ast.literal_eval to convert it to an actual list,
# then join as a readable comma-separated string.

def clean_amenities(a):
    if pd.isna(a): return None
    try:
        amenity_list = ast.literal_eval(str(a))   # String → actual Python list
        return ', '.join(amenity_list)              # List → 'A, B, C'
    except:
        return str(a)   # Fallback: keep original string

df3['amenities'] = df3['AMENITIES'].apply(clean_amenities)
print('Sample amenities cleaning:')
print(df3[['amenities']].head(3))

Sample amenities cleaning:
                                           amenities
0  Earthquake Resistant, Marbel, Parquet, Parking...
1  Earthquake Resistant, Parquet, Drinking Water,...
2  Earthquake Resistant, Marbel, Parquet, Parking...


In [118]:
# ── Step 10: Rename columns, extract district, drop raws ──────
df3 = df3.rename(columns={
    'TITLE'    : 'title',
    'LOCATION' : 'location_raw',
    'FLOOR'    : 'floors'
})

def extract_district(location):
    if pd.isna(location): return 'Unknown'
    location = str(location)
    for d in ['Kathmandu', 'Lalitpur', 'Bhaktapur']:
        if d.lower() in location.lower(): return d
    return 'Unknown'

df3['district'] = df3['location_raw'].apply(extract_district)

# Drop original raw columns
df3 = df3.drop(columns=[
    'PRICE', 'LAND AREA', 'BUILDUP AREA', 'ROAD ACCESS',
    'FACING', 'BUILT YEAR', 'PARKING', 'BEDROOM', 'BATHROOM', 'AMENITIES'
])

print('Final columns:', df3.columns.tolist())

Final columns: ['title', 'location_raw', 'floors', 'price_npr', 'land_size_aana', 'buildup_area_sqft', 'road_width_feet', 'facing', 'built_year_bs', 'built_year_ad', 'house_age', 'parking_cars', 'parking_bikes', 'bedrooms', 'bathrooms', 'amenities', 'district']


In [119]:
df3.sample(3)

,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district
25,House for Sale,"Sanepa, Lalitpur",2.5,63500000.0,5.0,NaN,10.0,North-West,2078.0,2021.0,4.0,NaN,NaN,5.0,3.0,"Earthquake Resistant, Drinking Water, Marbel, Parquet, Terrace, Parking, Modular Kitchen, MiCrowave, Bed Room, Dining Room, Guest Room",Lalitpur
2129,"House for sale near Real Banquet, Bafal","Bafal, Kathmandu",2.5,36500000.0,4.0,2400.0,12.0,South-West,2063.0,2006.0,19.0,NaN,NaN,7.0,3.0,"Drainage, Bathroom, Drinking Water, Parking",Kathmandu
780,"Beautiful house on sale at Sunakothi, Lalitpur","Sunakothi, Lalitpur",2.5,51500000.0,7.0,NaN,20.0,North,2072.0,2015.0,10.0,NaN,NaN,4.0,4.0,"Drinking Water, Bathroom, Drainage, Parking",Lalitpur


In [120]:
# ── Step 11: Final check ──────────────────────────────────────
print('=== FILE 3 FINAL SUMMARY ===')
print(f'Total rows: {len(df3)}')
print(f'Total columns: {len(df3.columns)}')
print('\nNull values per column:')
print(df3.isnull().sum())
df3.head()

=== FILE 3 FINAL SUMMARY ===
Total rows: 3228
Total columns: 17

Null values per column:
title                   0
location_raw            0
floors                 81
price_npr               0
land_size_aana         73
buildup_area_sqft    2562
road_width_feet        45
facing                177
built_year_bs          56
built_year_ad          56
house_age              56
parking_cars         2664
parking_bikes        2644
bedrooms              285
bathrooms             332
amenities               0
district                0
dtype: int64


,title,location_raw,floors,price_npr,land_size_aana,buildup_area_sqft,road_width_feet,facing,built_year_bs,built_year_ad,house_age,parking_cars,parking_bikes,bedrooms,bathrooms,amenities,district
0,House for Sale,"Imadol, Lalitpur",3.0,29000000.0,4.0,NaN,12.0,West,2076.0,2019.0,6.0,1.0,2.0,5.0,4.0,"Earthquake Resistant, Marbel, Parquet, Parking, Drinking Water, Terrace, Modular Kitchen, Closet, Dining Room, Pantry, Bed Room",Lalitpur
1,House for Sale,"Satdobato, Lalitpur",4.5,47500000.0,3.0,NaN,10.0,West,2076.0,2019.0,6.0,2.0,2.0,5.0,6.0,"Earthquake Resistant, Parquet, Drinking Water, Parking, Modular Kitchen, Closet, Bed Room, Pantry, Dining Room, Bathroom, Store Room",Lalitpur
2,4 BHK House for Sale,"Imadol, Lalitpur",2.5,19900000.0,2.3,NaN,10.0,West,2060.0,2003.0,22.0,1.0,3.0,4.0,4.0,"Earthquake Resistant, Marbel, Parquet, Parking, Drinking Water, Balcony, Closet, Modular Kitchen, MiCrowave, Bed Room, Dining Room",Lalitpur
3,Bungalow House for Sale,"Bhaisepati, Lalitpur",2.5,40000000.0,7.0,NaN,12.0,North-West,2059.0,2002.0,23.0,4.0,4.0,4.0,3.0,"Earthquake Resistant, Marbel, Parquet, Balcony, Parking, Drinking Water, Garden, Modular Kitchen, Internet, Closet, MiCrowave",Lalitpur
4,House for Rent,"Maharajgunj, Kathmandu",2.0,12000000.0,6.0,NaN,20.0,South,2071.0,2014.0,11.0,4.0,5.0,4.0,4.0,"Earthquake Resistant, Parquet, Parking, Drinking Water, Closet, Modular Kitchen, Bed Room, Bathroom, Living Room, Guest Room, Prayer Room",Kathmandu


In [121]:
# ── Step 12: Save cleaned file ────────────────────────────────
df3.to_csv('cleaned_nepali_houses.csv', index=False)
print('Saved: cleaned_nepali_houses.csv ✅')

Saved: cleaned_nepali_houses.csv ✅


---
# ✅ Summary

| File | Original Rows | After Cleaning | Key Changes |
|------|--------------|----------------|-------------|
| hamrobazaar_land | 3,869 | ~3,868 | Dropped Type, cleaned road col, removed 0-price rows |
| nepali_land | 2,043 | 2,043 | Price normalized with unit tag, road split into width+surface |
| nepali_houses | 3,418 | ~3,228 | Price standardized to NPR, all columns cleaned, outliers capped |

**Output files:**
- `cleaned_hamrobazaar_land.csv`
- `cleaned_nepali_land.csv`
- `cleaned_nepali_houses.csv`

**Next step:** Merge these into one master dataset!

In [ ]:
# ── Quick overview of all 3 cleaned files ─────────────────────
print('=== FINAL OVERVIEW ===')
print()
for name, df in [('cleaned_hamrobazaar_land', df1), ('cleaned_nepali_land', df2), ('cleaned_nepali_houses', df3)]:
    print(f'📄 {name}')
    print(f'   Rows: {len(df)} | Columns: {len(df.columns)}')
    print(f'   Columns: {df.columns.tolist()}')
    print()